# 📊 Master Time Series — Complete Theory Guide

> **Scope**: This notebook is a comprehensive, code-free theory reference for **Time Series Analysis & Forecasting**. It covers decomposition, stationarity testing, feature engineering, classical models (ARIMA/SARIMA), evaluation metrics, and modern ML-based forecasting. Write your own code cells as you study each section.

---

## Table of Contents

1. [What is a Time Series?](#1)
2. [Time Series Components](#2)
3. [Additive vs Multiplicative Decomposition](#3)
4. [Classical Decomposition vs STL](#4)
5. [Stationarity — Concept & Importance](#5)
6. [Augmented Dickey-Fuller (ADF) Test](#6)
7. [KPSS Test](#7)
8. [ADF + KPSS Decision Matrix](#8)
9. [Differencing — Making Series Stationary](#9)
10. [Log & Power Transforms](#10)
11. [Lag Features](#11)
12. [Rolling Window Statistics](#12)
13. [Expanding Window Statistics](#13)
14. [Time-Based Dummy Variables](#14)
15. [Walk-Forward Cross-Validation](#15)
16. [Autocorrelation Function (ACF)](#16)
17. [Partial Autocorrelation Function (PACF)](#17)
18. [ACF & PACF — Identifying Model Orders](#18)
19. [Autoregressive Model — AR(p)](#19)
20. [Moving Average Model — MA(q)](#20)
21. [ARMA(p, q) Model](#21)
22. [Integration & Differencing — I(d)](#22)
23. [ARIMA(p, d, q) Model](#23)
24. [Seasonal ARIMA — SARIMA](#24)
25. [Box-Jenkins Methodology](#25)
26. [Auto-ARIMA (pmdarima)](#26)
27. [Forecasting Evaluation Metrics](#27)
28. [ML-Based Time Series Forecasting](#28)
29. [Facebook Prophet](#29)
30. [ARIMA vs ML — Comparison Matrix](#30)
31. [Key Takeaways & Interview Questions](#31)

---

<a id='1'></a>
## 1. What is a Time Series?

A **time series** is a sequence of data points collected or recorded at **successive, equally-spaced points in time**.

$$\{y_1, y_2, y_3, \ldots, y_T\} \quad \text{where } y_t \text{ is the observation at time } t$$

### Key Characteristics

| Property | Description | Example |
|----------|-------------|--------|
| **Temporal Ordering** | Data points have a natural time-based order | Stock prices day-by-day |
| **Autocorrelation** | Current values depend on past values | Today's temperature ≈ yesterday's |
| **Non-independence** | Observations are NOT i.i.d. (independent, identically distributed) | Unlike tabular ML data |
| **Frequency** | Regular intervals: hourly, daily, weekly, monthly, yearly | Monthly retail sales |

### Time Series vs Cross-Sectional Data

| Aspect | Time Series | Cross-Sectional (Tabular) |
|--------|------------|-------------------------|
| **Order** | Order matters (temporal dependency) | Order doesn't matter |
| **Independence** | Observations are correlated | Observations are assumed i.i.d. |
| **Train/Test Split** | Must respect temporal order | Random split is fine |
| **Goal** | Forecast future values | Predict target variable |

### Common Applications
- 📈 Stock price prediction
- 🌡️ Weather forecasting
- 🛒 Demand forecasting (retail, supply chain)
- 🏥 Patient health monitoring (ECG, EEG)
- ⚡ Energy consumption prediction
- 💰 Revenue/Sales forecasting

---

<a id='2'></a>
## 2. Time Series Components

Every time series can be decomposed into **four fundamental components**:

### 2.1 Trend ($T_t$)
The **long-term directional movement** in the data — upward, downward, or flat.

- Represents the underlying growth or decline over time
- Can be **linear** ($T_t = a + bt$) or **non-linear** (polynomial, exponential)
- Example: GDP growth over decades, population increase

### 2.2 Seasonality ($S_t$)
**Fixed-period, repeating patterns** that occur at regular intervals.

- Period is **known and constant** (e.g., 12 months, 7 days, 24 hours)
- Caused by calendar effects, weather, holidays, business cycles
- Example: Ice cream sales peak every summer (period = 12 months)

### 2.3 Cyclical ($C_t$)
**Non-fixed period oscillations** — similar to seasonality but with **variable, longer periods**.

- Period is **unknown and irregular** (typically 2-10 years)
- Driven by economic/business cycles, market sentiments
- Example: Business expansion and recession cycles
- **Note**: Often combined with Trend in practice (Trend-Cycle component)

### 2.4 Residual / Noise / Irregular ($R_t$ or $\epsilon_t$)
**Random, unpredictable fluctuations** remaining after removing Trend, Seasonality, and Cyclical components.

- Should ideally be **white noise** (zero mean, constant variance, no autocorrelation)
- If residuals show patterns → model hasn't captured all structure
- Example: Unexpected market shocks, natural disasters

### Visual Summary

```
Time Series = Trend + Seasonality + Cyclical + Residual
     y_t    =  T_t  +    S_t     +   C_t    +   R_t
```

---

<a id='3'></a>
## 3. Additive vs Multiplicative Decomposition

### 3.1 Additive Model

$$\boxed{Y_t = T_t + S_t + R_t}$$

**When to use**: Seasonal fluctuations remain **constant in magnitude** regardless of the level of the series.

- Seasonal swings are roughly the **same absolute size** over time
- Example: Temperature varies by ±10°C every year, regardless of the long-term trend

### 3.2 Multiplicative Model

$$\boxed{Y_t = T_t \times S_t \times R_t}$$

**When to use**: Seasonal fluctuations **scale proportionally** with the level of the series.

- As the trend increases, seasonal swings get **larger in absolute terms**
- Can be converted to additive by taking **log transform**: $\log(Y_t) = \log(T_t) + \log(S_t) + \log(R_t)$
- Example: Airline passenger numbers — seasonal peaks grow as overall traffic grows

### 3.3 How to Choose?

| Criterion | Additive | Multiplicative |
|-----------|----------|----------------|
| **Seasonal amplitude** | Constant over time | Grows/shrinks with trend |
| **Visual check** | Parallel seasonal bands | Widening/narrowing bands |
| **Data with zeros** | ✅ Works fine | ❌ Cannot have zeros |
| **Log transform trick** | Not needed | $\log(Y_t)$ converts to additive |
| **Common in** | Temperature, some industrial processes | Sales, finance, demographics |

### 3.4 Pseudo-Additive Model (Bonus)

$$Y_t = T_t \times (S_t + R_t - 1)$$

Used when the series has zeros or very small values but multiplicative seasonality — a hybrid approach.

---

<a id='4'></a>
## 4. Classical Decomposition vs STL

### 4.1 Classical Decomposition

Uses **moving averages** to estimate the trend, then extracts seasonality from the detrended series.

**Steps (Additive)**:
1. **Estimate Trend** $\hat{T}_t$: Apply centered moving average of order $m$ (= seasonal period)
   $$\hat{T}_t = \frac{1}{m} \sum_{j=-k}^{k} y_{t+j} \quad \text{where } m = 2k+1$$
   For even $m$ (e.g., $m=12$): use $2 \times m$-MA (average of two $m$-MAs)
2. **Detrend**: $y_t - \hat{T}_t$
3. **Estimate Seasonality** $\hat{S}_t$: Average the detrended values for each seasonal period
4. **Residual**: $\hat{R}_t = y_t - \hat{T}_t - \hat{S}_t$

**Limitations**:
- ❌ Trend estimate unavailable for first and last $k$ observations
- ❌ Assumes seasonal component is exactly repeating (no evolution)
- ❌ Sensitive to outliers
- ❌ Only handles additive or multiplicative, not mixed

### 4.2 STL Decomposition (Seasonal and Trend decomposition using Loess)

Uses **locally weighted regression (LOESS)** to estimate trend and seasonality.

**Advantages over Classical**:
- ✅ Handles **any type of seasonality** (not just monthly/quarterly)
- ✅ Seasonal component **can change over time** (controlled by `seasonal` window)
- ✅ **Robust mode** available — resistant to outliers
- ✅ Trend smoothness controllable via `trend` window parameter
- ✅ No missing values at boundaries

**Key Parameters**:

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `period` | Known seasonal period | Must be specified correctly |
| `seasonal` | Window size for seasonal extraction | Larger = more stable seasonal pattern |
| `trend` | Window size for trend extraction | Larger = smoother trend |
| `robust` | Use robust fitting (bisquare weights) | True = resistant to outliers |

**Python**: `from statsmodels.tsa.seasonal import STL`

### 4.3 MSTL (Multiple Seasonal-Trend decomposition using Loess)

Extension of STL for series with **multiple seasonal periods** (e.g., daily data with weekly AND yearly seasonality).

$$Y_t = T_t + S_t^{(1)} + S_t^{(2)} + \ldots + S_t^{(K)} + R_t$$

---

<a id='5'></a>
## 5. Stationarity — Concept & Importance

### 5.1 Why Stationarity Matters

Most classical time series models (AR, MA, ARIMA) **assume stationarity**. A non-stationary series has statistical properties that change over time, making it impossible to generalize patterns learned from the past.

### 5.2 Strict (Strong) Stationarity

The **joint probability distribution** of any collection of time points is invariant under time shifts:

$$F(y_{t_1}, y_{t_2}, \ldots, y_{t_k}) = F(y_{t_1+h}, y_{t_2+h}, \ldots, y_{t_k+h}) \quad \forall h, k$$

This is very restrictive and rarely tested in practice.

### 5.3 Weak (Wide-Sense) Stationarity

Three conditions must hold:

**Condition 1 — Constant Mean**:
$$E[Y_t] = \mu \quad \text{for all } t$$

**Condition 2 — Constant Variance**:
$$\text{Var}(Y_t) = \sigma^2 \quad \text{for all } t$$

**Condition 3 — Autocovariance depends only on lag**:
$$\text{Cov}(Y_t, Y_{t+h}) = \gamma(h) \quad \text{(function of lag } h \text{ only, not } t\text{)}$$

### 5.4 Common Causes of Non-Stationarity

| Cause | Effect | Solution |
|-------|--------|----------|
| **Trend** | Mean changes over time | Differencing or detrending |
| **Seasonality** | Periodic patterns | Seasonal differencing |
| **Changing variance** | Variance grows/shrinks | Log/Box-Cox transform |
| **Structural breaks** | Sudden regime change | Segment the series |
| **Unit root** | Random walk behavior | Differencing |

### 5.5 Unit Root

A **unit root** exists when the characteristic equation of the autoregressive model has a root equal to 1.

Consider: $y_t = \phi y_{t-1} + \epsilon_t$

- If $|\phi| < 1$: **Stationary** (mean-reverting)
- If $\phi = 1$: **Unit root / Random Walk** → Non-stationary
- If $|\phi| > 1$: **Explosive** → Non-stationary

Random Walk: $y_t = y_{t-1} + \epsilon_t$ → variance grows linearly with time: $\text{Var}(y_t) = t \cdot \sigma^2_\epsilon$

---

<a id='6'></a>
## 6. Augmented Dickey-Fuller (ADF) Test

### 6.1 The Dickey-Fuller Test

Tests for a **unit root** in an AR(1) process.

**Model**: $y_t = \phi y_{t-1} + \epsilon_t$

Reparameterize: $\Delta y_t = (\phi - 1) y_{t-1} + \epsilon_t = \gamma y_{t-1} + \epsilon_t$

where $\gamma = \phi - 1$

- $H_0: \gamma = 0$ (unit root exists → **non-stationary**)
- $H_1: \gamma < 0$ (no unit root → **stationary**)

### 6.2 The Augmented Dickey-Fuller (ADF) Test

Extends DF to handle **higher-order autocorrelation** by adding lagged difference terms:

$$\boxed{\Delta y_t = \alpha + \beta t + \gamma y_{t-1} + \sum_{i=1}^{p} \delta_i \Delta y_{t-i} + \epsilon_t}$$

| Term | Purpose |
|------|--------|
| $\alpha$ | Drift / intercept term |
| $\beta t$ | Deterministic trend |
| $\gamma y_{t-1}$ | **Unit root test parameter** |
| $\sum \delta_i \Delta y_{t-i}$ | Lagged differences to handle autocorrelation |
| $\epsilon_t$ | White noise error |

### 6.3 Hypotheses

- **$H_0: \gamma = 0$** → Unit root present → **Series is NON-STATIONARY**
- **$H_1: \gamma < 0$** → No unit root → **Series is STATIONARY**

### 6.4 Interpreting Results

| ADF Statistic vs Critical Values | p-value | Conclusion |
|-----------------------------------|---------|------------|
| ADF stat < critical value (more negative) | p ≤ 0.05 | **Reject $H_0$** → Stationary ✅ |
| ADF stat > critical value (less negative) | p > 0.05 | **Fail to reject $H_0$** → Non-Stationary ❌ |

### 6.5 Lag Selection

The number of lagged difference terms $p$ is chosen using:
- **AIC** (Akaike Information Criterion) — `autolag='AIC'` (default)
- **BIC** (Bayesian Information Criterion) — `autolag='BIC'`

**Python**: `from statsmodels.tsa.stattools import adfuller`

---

<a id='7'></a>
## 7. KPSS Test (Kwiatkowski–Phillips–Schmidt–Shin)

### 7.1 Key Difference from ADF

KPSS has the **opposite null hypothesis** compared to ADF:

- **$H_0$: Series IS stationary** (trend-stationary or level-stationary)
- **$H_1$: Series is NOT stationary** (has a unit root)

### 7.2 KPSS Model

$$y_t = \xi t + r_t + \epsilon_t$$

where $r_t = r_{t-1} + u_t$ is a random walk and $u_t \sim \text{iid}(0, \sigma^2_u)$

- **$H_0$**: $\sigma^2_u = 0$ (no random walk component → stationary)
- **$H_1$**: $\sigma^2_u > 0$ (random walk present → non-stationary)

### 7.3 Two Variants

| Variant | `regression=` | Tests for |
|---------|---------------|----------|
| **Level stationary** | `'c'` | Stationarity around a constant mean |
| **Trend stationary** | `'ct'` | Stationarity around a deterministic trend |

### 7.4 Interpreting Results

| KPSS Statistic | p-value | Conclusion |
|----------------|---------|------------|
| Stat < critical value | p > 0.05 | **Fail to reject $H_0$** → Stationary ✅ |
| Stat > critical value | p ≤ 0.05 | **Reject $H_0$** → Non-Stationary ❌ |

**Python**: `from statsmodels.tsa.stattools import kpss`

---

<a id='8'></a>
## 8. ADF + KPSS Decision Matrix

Using **both tests together** provides a more robust stationarity diagnosis:

| ADF Result | KPSS Result | Conclusion | Action |
|------------|-------------|------------|--------|
| Reject $H_0$ (p<0.05) | Fail to reject $H_0$ (p>0.05) | **Stationary** ✅ | No transformation needed |
| Fail to reject $H_0$ | Reject $H_0$ | **Non-Stationary** ❌ | Apply differencing |
| Reject $H_0$ | Reject $H_0$ | **Trend-Stationary** ⚠️ | Remove deterministic trend |
| Fail to reject $H_0$ | Fail to reject $H_0$ | **Inconclusive** 🤔 | Increase sample size or try different lags |

### Practical Diagnostic Workflow

```
1. Plot the series → Visual inspection
2. Plot rolling mean & rolling std → Check for drift
3. Run ADF test
4. Run KPSS test
5. Cross-reference with decision matrix above
6. If non-stationary → Apply transformations (differencing, log, etc.)
7. Re-test after transformation
```

---

<a id='9'></a>
## 9. Differencing — Making Series Stationary

### 9.1 First-Order Differencing ($d = 1$)

$$\Delta y_t = y_t - y_{t-1}$$

Removes **linear trend**. Most common transformation.

### 9.2 Second-Order Differencing ($d = 2$)

$$\Delta^2 y_t = \Delta(\Delta y_t) = (y_t - y_{t-1}) - (y_{t-1} - y_{t-2}) = y_t - 2y_{t-1} + y_{t-2}$$

Removes **quadratic trend**. Rarely needed (if $d > 2$, reconsider your model).

### 9.3 Seasonal Differencing ($D = 1$)

$$\Delta_m y_t = y_t - y_{t-m}$$

where $m$ is the seasonal period (e.g., $m = 12$ for monthly data).

Removes **seasonal patterns**.

### 9.4 Combined Differencing

For data with both trend AND seasonality, apply both:

$$\Delta \Delta_m y_t = (y_t - y_{t-m}) - (y_{t-1} - y_{t-1-m})$$

### 9.5 Over-Differencing Warning ⚠️

- **Under-differencing**: Residuals still have patterns → need more differencing
- **Over-differencing**: Introduces **artificial negative autocorrelation** → undo one level
- **Rule of thumb**: Variance should **decrease** after differencing. If it increases → you've over-differenced.

### 9.6 Backshift Operator Notation (B)

Compact mathematical notation used in ARIMA literature:

$$B \cdot y_t = y_{t-1}$$
$$B^k \cdot y_t = y_{t-k}$$
$$(1 - B) y_t = y_t - y_{t-1} = \Delta y_t$$
$$(1 - B^m) y_t = y_t - y_{t-m} = \Delta_m y_t$$

---

<a id='10'></a>
## 10. Log & Power Transforms

### 10.1 Log Transform

$$z_t = \ln(y_t)$$

**Purpose**: Stabilize **variance** that grows proportionally with the level.

- Converts multiplicative model to additive: $\ln(T \times S \times R) = \ln(T) + \ln(S) + \ln(R)$
- Requires all values $y_t > 0$
- Common in financial data, population data

### 10.2 Box-Cox Transform

$$z_t = \begin{cases} \frac{y_t^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(y_t) & \text{if } \lambda = 0 \end{cases}$$

| $\lambda$ | Transform |
|-----------|----------|
| 1.0 | No transform (identity) |
| 0.5 | Square root |
| 0.0 | Natural log |
| -0.5 | Reciprocal square root |
| -1.0 | Reciprocal |

Optimal $\lambda$ can be found via **maximum likelihood estimation**.

### 10.3 Yeo-Johnson Transform

Extension of Box-Cox that works with **negative values** and **zeros**.

**Python**: `from scipy.stats import boxcox, yeojohnson`

---

<a id='11'></a>
## 11. Lag Features

### 11.1 Concept

Use **past values** of the target variable as input features for a model:

$$\hat{y}_t = f(y_{t-1}, y_{t-2}, \ldots, y_{t-k})$$

### 11.2 Creating Lag Features

| Feature | Formula | Description |
|---------|---------|-------------|
| `lag_1` | $y_{t-1}$ | Previous time step |
| `lag_2` | $y_{t-2}$ | Two steps ago |
| `lag_7` | $y_{t-7}$ | Same day last week (daily data) |
| `lag_12` | $y_{t-12}$ | Same month last year (monthly data) |
| `lag_365` | $y_{t-365}$ | Same day last year (daily data) |

### 11.3 How Many Lags?

- Use **ACF/PACF plots** to identify significant lags (covered in Section 16-17)
- Domain knowledge: use lags that correspond to known seasonal periods
- More lags = more features but more NaN rows at the start

### 11.4 Difference Features

Instead of raw lags, use **differences between lags**:

| Feature | Formula | Captures |
|---------|---------|----------|
| `diff_1` | $y_t - y_{t-1}$ | Short-term change |
| `diff_7` | $y_t - y_{t-7}$ | Week-over-week change |
| `pct_change_1` | $(y_t - y_{t-1}) / y_{t-1}$ | Percentage change |

**Python**: `df['lag_1'] = df['y'].shift(1)`

---

<a id='12'></a>
## 12. Rolling Window Statistics

### 12.1 Concept

Compute statistics over a **fixed-size sliding window** of the last $w$ observations:

### 12.2 Common Rolling Features

| Feature | Formula | Purpose |
|---------|---------|--------|
| **Rolling Mean** | $\bar{y}_t^{(w)} = \frac{1}{w} \sum_{i=0}^{w-1} y_{t-i}$ | Smoothed trend, reduces noise |
| **Rolling Std** | $\sigma_t^{(w)} = \sqrt{\frac{1}{w-1} \sum_{i=0}^{w-1} (y_{t-i} - \bar{y}_t)^2}$ | Volatility measure |
| **Rolling Min** | $\min(y_{t}, y_{t-1}, \ldots, y_{t-w+1})$ | Support level |
| **Rolling Max** | $\max(y_{t}, y_{t-1}, \ldots, y_{t-w+1})$ | Resistance level |
| **Rolling Median** | Median of last $w$ values | Robust central tendency |
| **Rolling Skew** | Skewness of last $w$ values | Distribution asymmetry |

### 12.3 Exponentially Weighted Moving Average (EWMA)

Unlike simple rolling average, EWMA gives **more weight to recent observations**:

$$\text{EWMA}_t = \alpha \cdot y_t + (1 - \alpha) \cdot \text{EWMA}_{t-1}$$

where $\alpha \in (0, 1)$ is the **smoothing factor** (higher = more responsive to recent data).

Alternatively, specify `span` $= \frac{2}{\alpha} - 1$ or `halflife`.

### 12.4 Window Size Selection

- **Small window** (e.g., 3-7): More responsive, noisier features
- **Large window** (e.g., 30-90): Smoother, captures longer trends
- **Match to seasonality**: Window = seasonal period often works well

**Python**: `df['rolling_mean_7'] = df['y'].rolling(window=7).mean()`

---

<a id='13'></a>
## 13. Expanding Window Statistics

### 13.1 Concept

Unlike rolling windows (fixed size), expanding windows include **all observations from the start up to time $t$**:

$$\bar{y}_t^{\text{expanding}} = \frac{1}{t} \sum_{i=1}^{t} y_i$$

### 13.2 Expanding vs Rolling

| Aspect | Rolling Window | Expanding Window |
|--------|---------------|------------------|
| **Window size** | Fixed ($w$ observations) | Grows over time |
| **Memory** | Forgets old data | Remembers all history |
| **Sensitivity** | More responsive to recent changes | More stable, less noisy |
| **Use case** | Short-term patterns | Cumulative statistics |

### 13.3 Common Expanding Features

- **Expanding mean**: Cumulative average up to $t$
- **Expanding std**: Cumulative standard deviation
- **Expanding min/max**: All-time low/high up to $t$
- **Expanding count**: Count of non-null observations

**Python**: `df['expanding_mean'] = df['y'].expanding().mean()`

---

<a id='14'></a>
## 14. Time-Based Dummy Variables

### 14.1 Concept

Extract **calendar features** from the datetime index to capture seasonal and cyclical patterns:

### 14.2 Common Calendar Features

| Feature | Values | Captures |
|---------|--------|----------|
| `hour` | 0-23 | Intra-day patterns |
| `day_of_week` | 0-6 (Mon-Sun) | Weekly seasonality |
| `day_of_month` | 1-31 | Monthly patterns (e.g., paydays) |
| `day_of_year` | 1-365 | Yearly position |
| `week_of_year` | 1-52 | Weekly position in year |
| `month` | 1-12 | Monthly seasonality |
| `quarter` | 1-4 | Quarterly patterns |
| `year` | e.g., 2024 | Long-term trend proxy |
| `is_weekend` | 0/1 | Weekend vs weekday |
| `is_holiday` | 0/1 | Public holidays |
| `is_month_start` | 0/1 | Start-of-month effects |
| `is_month_end` | 0/1 | End-of-month effects |

### 14.3 Cyclical Encoding

For features with **cyclical nature** (e.g., hour, month), linear encoding creates artificial discontinuities (e.g., December=12 is far from January=1). Use **sine/cosine encoding**:

$$x_{\sin} = \sin\left(\frac{2\pi \cdot v}{\text{max\_val}}\right) \qquad x_{\cos} = \cos\left(\frac{2\pi \cdot v}{\text{max\_val}}\right)$$

Example for months (max_val = 12):
- January (v=1): $\sin(2\pi/12) = 0.5$, $\cos(2\pi/12) = 0.866$
- December (v=12): $\sin(24\pi/12) = 0$, $\cos(24\pi/12) = 1$

This ensures December and January are **close** in feature space.

### 14.4 Holiday Features

- Use Python `holidays` library for country-specific holidays
- Can add **days before/after holiday** as separate features
- Event-specific dummies (Black Friday, Diwali, etc.)

---

<a id='15'></a>
## 15. Walk-Forward Cross-Validation

### 15.1 Why Regular K-Fold Fails for Time Series

Standard K-Fold CV **randomly shuffles** data, destroying temporal order. This causes:
- **Data leakage**: Model trains on future data and predicts past → overly optimistic scores
- **Broken autocorrelation**: Assumes i.i.d. which time series violates

### 15.2 Walk-Forward (Expanding Window) Validation

```
Fold 1: [TRAIN: ████████] [TEST: ██]
Fold 2: [TRAIN: ██████████] [TEST: ██]
Fold 3: [TRAIN: ████████████] [TEST: ██]
Fold 4: [TRAIN: ██████████████] [TEST: ██]
```

- Training set **expands** with each fold
- Test set always comes **after** training set temporally
- Mimics real-world scenario: train on past → predict future

### 15.3 Sliding Window Validation

```
Fold 1: [TRAIN: ████████] [TEST: ██]
Fold 2:   [TRAIN: ████████] [TEST: ██]
Fold 3:     [TRAIN: ████████] [TEST: ██]
Fold 4:       [TRAIN: ████████] [TEST: ██]
```

- Training window is **fixed size** (slides forward)
- Better when old data is less relevant (concept drift)

### 15.4 Gap Between Train and Test

Sometimes add a **gap** between train and test to prevent leakage from lag features:

```
Fold: [TRAIN: ████████] [GAP: ░░] [TEST: ██]
```

Gap size should equal the **maximum lag** used in feature engineering.

### 15.5 Python Implementation

- **sklearn**: `TimeSeriesSplit(n_splits=5)`
- This implements **expanding window** by default
- For sliding window: set `max_train_size` parameter
- For gap: set `gap` parameter

---

<a id='16'></a>
## 16. Autocorrelation Function (ACF)

### 16.1 Definition

The ACF measures the **correlation between a time series and its own lagged values**:

$$\boxed{\rho(h) = \frac{\text{Cov}(Y_t, Y_{t+h})}{\text{Var}(Y_t)} = \frac{\gamma(h)}{\gamma(0)}}$$

where:
- $\gamma(h) = \text{Cov}(Y_t, Y_{t+h})$ = autocovariance at lag $h$
- $\gamma(0) = \text{Var}(Y_t)$ = variance (autocovariance at lag 0)
- $\rho(0) = 1$ always (series is perfectly correlated with itself)
- $-1 \leq \rho(h) \leq 1$

### 16.2 ACF Plot (Correlogram)

Shows $\rho(h)$ as vertical bars for lags $h = 0, 1, 2, \ldots, H$

- **Blue shaded region**: 95% confidence interval ($\approx \pm 1.96 / \sqrt{n}$)
- Bars **outside** the shaded region → **statistically significant** autocorrelation

### 16.3 ACF Patterns for Common Processes

| Process | ACF Behavior |
|---------|-------------|
| **White Noise** | All lags ≈ 0 (within confidence bands) |
| **AR(p)** | Decays **exponentially** or in **damped oscillations** |
| **MA(q)** | **Cuts off sharply** after lag $q$ |
| **ARMA** | Decays exponentially (combined AR+MA behavior) |
| **Non-Stationary** | Slow, linear decay → needs differencing |
| **Seasonal** | Significant spikes at seasonal lags ($m, 2m, 3m, \ldots$) |

### 16.4 Important Note: ACF includes INDIRECT correlations

ACF at lag 2 includes the correlation that flows **through lag 1**:

$y_t \xleftrightarrow{\text{direct}} y_{t-1} \xleftrightarrow{\text{direct}} y_{t-2}$ → ACF(2) captures both direct AND indirect effects.

This is why we need PACF (next section).

**Python**: `from statsmodels.graphics.tsaplots import plot_acf`

---

<a id='17'></a>
## 17. Partial Autocorrelation Function (PACF)

### 17.1 Definition

PACF measures the **direct correlation** between $Y_t$ and $Y_{t-h}$ **after removing the effects of all intermediate lags** ($Y_{t-1}, Y_{t-2}, \ldots, Y_{t-h+1}$).

$$\boxed{\phi_{hh} = \text{Corr}(Y_t, Y_{t-h} \mid Y_{t-1}, Y_{t-2}, \ldots, Y_{t-h+1})}$$

### 17.2 How PACF is Computed

Fit successive AR models of increasing order:

1. Fit AR(1): $Y_t = \phi_{11} Y_{t-1} + \epsilon_t$ → $\phi_{11}$ = PACF at lag 1
2. Fit AR(2): $Y_t = \phi_{21} Y_{t-1} + \phi_{22} Y_{t-2} + \epsilon_t$ → $\phi_{22}$ = PACF at lag 2
3. And so on...

The PACF at lag $h$ is the **last coefficient** $\phi_{hh}$ in an AR(h) model.

### 17.3 PACF Patterns

| Process | PACF Behavior |
|---------|---------------|
| **White Noise** | All lags ≈ 0 |
| **AR(p)** | **Cuts off sharply** after lag $p$ |
| **MA(q)** | Decays **exponentially** or in damped oscillations |
| **ARMA** | Exponential decay (no sharp cutoff) |

### 17.4 Key Insight: ACF vs PACF

| | ACF | PACF |
|--|-----|------|
| **Includes indirect correlations** | ✅ Yes | ❌ No (removed) |
| **Identifies AR order** | Decays gradually | **Cuts off at lag p** |
| **Identifies MA order** | **Cuts off at lag q** | Decays gradually |
| **Useful for** | Determining MA(q) | Determining AR(p) |

**Python**: `from statsmodels.graphics.tsaplots import plot_pacf`

---

<a id='18'></a>
## 18. ACF & PACF — Identifying Model Orders

### Decision Table for Model Selection

| ACF Pattern | PACF Pattern | Suggested Model |
|-------------|-------------|------------------|
| Exponential decay or damped oscillation | Cuts off after lag $p$ | **AR(p)** |
| Cuts off after lag $q$ | Exponential decay or damped oscillation | **MA(q)** |
| Exponential decay | Exponential decay | **ARMA(p, q)** — try small $p, q$ |
| Slow linear decay | Large first spike | **Non-stationary** — difference first |
| Significant spikes at seasonal lags | Significant spikes at seasonal lags | **Seasonal** component needed |

### Example: ACF/PACF of AR(2)

```
ACF:  ████ ███ ██ █ ░ ░ ░ ░    (gradual decay)
PACF: ████ ███ ░ ░ ░ ░ ░ ░    (cuts off after lag 2)
       L0   L1  L2 L3 L4 L5
```

→ PACF cuts off at lag 2 → Suggests **AR(2)**

### Example: ACF/PACF of MA(1)

```
ACF:  ████ ███ ░ ░ ░ ░ ░ ░    (cuts off after lag 1)
PACF: ████ ███ ██ █ ░ ░ ░ ░    (gradual decay)
       L0   L1  L2 L3 L4 L5
```

→ ACF cuts off at lag 1 → Suggests **MA(1)**

---

<a id='19'></a>
## 19. Autoregressive Model — AR(p)

### 19.1 Definition

An **Autoregressive model of order $p$** predicts the current value as a **linear combination of $p$ past values**:

$$\boxed{Y_t = c + \phi_1 Y_{t-1} + \phi_2 Y_{t-2} + \ldots + \phi_p Y_{t-p} + \epsilon_t}$$

where:
- $c$ = constant (intercept)
- $\phi_1, \phi_2, \ldots, \phi_p$ = autoregressive coefficients
- $\epsilon_t \sim \text{WN}(0, \sigma^2)$ = white noise error

### 19.2 Using Backshift Operator

$$\Phi(B) Y_t = c + \epsilon_t$$

where $\Phi(B) = 1 - \phi_1 B - \phi_2 B^2 - \ldots - \phi_p B^p$ is the **AR polynomial**.

### 19.3 Stationarity Conditions

AR(p) is stationary when the **roots of the characteristic polynomial** $\Phi(z) = 0$ all lie **outside the unit circle** ($|z| > 1$).

For AR(1): Stationary if $|\phi_1| < 1$

### 19.4 AR(1) Properties

| Property | Formula |
|----------|--------|
| **Mean** | $\mu = \frac{c}{1 - \phi_1}$ |
| **Variance** | $\sigma^2_Y = \frac{\sigma^2_\epsilon}{1 - \phi_1^2}$ |
| **ACF** | $\rho(h) = \phi_1^h$ (exponential decay) |
| **PACF** | $\phi_{11} = \phi_1$, $\phi_{hh} = 0$ for $h > 1$ |

### 19.5 AR(2) Properties

- ACF: Can show **damped oscillations** if $\phi_1^2 + 4\phi_2 < 0$ (complex roots)
- Stationarity requires: $\phi_2 + \phi_1 < 1$, $\phi_2 - \phi_1 < 1$, $|\phi_2| < 1$

### 19.6 How to Choose $p$

- **PACF**: Cuts off after lag $p$ → order is $p$
- **AIC/BIC**: Compare models with different $p$ values, choose lowest information criterion

---

<a id='20'></a>
## 20. Moving Average Model — MA(q)

### 20.1 Definition

A **Moving Average model of order $q$** predicts the current value as a **linear combination of $q$ past error terms**:

$$\boxed{Y_t = \mu + \epsilon_t + \theta_1 \epsilon_{t-1} + \theta_2 \epsilon_{t-2} + \ldots + \theta_q \epsilon_{t-q}}$$

where:
- $\mu$ = mean of the series
- $\theta_1, \theta_2, \ldots, \theta_q$ = moving average coefficients
- $\epsilon_t \sim \text{WN}(0, \sigma^2)$ = white noise error

### 20.2 Using Backshift Operator

$$Y_t = \mu + \Theta(B) \epsilon_t$$

where $\Theta(B) = 1 + \theta_1 B + \theta_2 B^2 + \ldots + \theta_q B^q$ is the **MA polynomial**.

### 20.3 Key Properties

- **Always stationary** for any finite $q$ (regardless of coefficient values)
- **Invertibility** condition: Roots of $\Theta(z) = 0$ lie outside unit circle (needed for unique representation)

### 20.4 MA(1) Properties

| Property | Formula |
|----------|--------|
| **Mean** | $E[Y_t] = \mu$ |
| **Variance** | $\text{Var}(Y_t) = (1 + \theta_1^2) \sigma^2_\epsilon$ |
| **ACF at lag 1** | $\rho(1) = \frac{\theta_1}{1 + \theta_1^2}$ |
| **ACF at lag h > 1** | $\rho(h) = 0$ (**sharp cutoff**) |

### 20.5 How to Choose $q$

- **ACF**: Cuts off after lag $q$ → order is $q$
- **AIC/BIC**: Compare models with different $q$ values

### 20.6 AR vs MA — Key Distinction

| | AR(p) | MA(q) |
|-|-------|-------|
| **Depends on** | Past **values** $Y_{t-1}, \ldots$ | Past **errors** $\epsilon_{t-1}, \ldots$ |
| **ACF** | Exponential decay | **Cuts off** at lag $q$ |
| **PACF** | **Cuts off** at lag $p$ | Exponential decay |
| **Stationarity** | Requires conditions on $\phi_i$ | **Always** stationary |
| **Memory** | Infinite (through recursive structure) | Finite ($q$ steps) |

---

<a id='21'></a>
## 21. ARMA(p, q) Model

### 21.1 Definition

Combines **AR and MA** components:

$$\boxed{Y_t = c + \sum_{i=1}^{p} \phi_i Y_{t-i} + \epsilon_t + \sum_{j=1}^{q} \theta_j \epsilon_{t-j}}$$

Using backshift: $\Phi(B) Y_t = c + \Theta(B) \epsilon_t$

### 21.2 Properties

- **ACF**: Gradual decay (no sharp cutoff) — mixture of AR and MA patterns
- **PACF**: Gradual decay (no sharp cutoff)
- Requires **stationarity** (AR roots outside unit circle)
- Requires **invertibility** (MA roots outside unit circle)

### 21.3 Model Selection for ARMA

Since neither ACF nor PACF has a clean cutoff, use **information criteria**:

$$\text{AIC} = -2 \ln(L) + 2k \qquad \text{BIC} = -2 \ln(L) + k \ln(n)$$

where $L$ = likelihood, $k$ = number of parameters, $n$ = sample size.

- AIC: Better for **prediction** (slight tendency to overfit)
- BIC: Better for **model identification** (penalizes complexity more)

---

<a id='22'></a>
## 22. Integration & Differencing — I(d)

### 22.1 Concept

If a series becomes stationary after $d$ differencing operations, it is said to be **integrated of order $d$**, denoted $I(d)$.

| Notation | Meaning | Example |
|----------|---------|--------|
| $I(0)$ | Already stationary | White noise |
| $I(1)$ | Stationary after **one** differencing | Random walk |
| $I(2)$ | Stationary after **two** differencings | Quadratic trend |

### 22.2 Random Walk = I(1)

$Y_t = Y_{t-1} + \epsilon_t$ (AR(1) with $\phi = 1$)

After first differencing: $\Delta Y_t = Y_t - Y_{t-1} = \epsilon_t$ (white noise → stationary)

### 22.3 Determining $d$

1. Start with the raw series → Run ADF test
2. If non-stationary → Apply first difference ($d = 1$) → Re-test
3. If still non-stationary → Apply second difference ($d = 2$) → Re-test
4. Typically $d \leq 2$ is sufficient

---

<a id='23'></a>
## 23. ARIMA(p, d, q) Model

### 23.1 Definition

**AutoRegressive Integrated Moving Average** — extends ARMA by incorporating differencing:

$$\boxed{\Phi(B)(1 - B)^d Y_t = c + \Theta(B) \epsilon_t}$$

Expanded:

$$\Delta^d Y_t = c + \sum_{i=1}^{p} \phi_i \Delta^d Y_{t-i} + \epsilon_t + \sum_{j=1}^{q} \theta_j \epsilon_{t-j}$$

### 23.2 Parameters

| Parameter | Name | Range | Determined By |
|-----------|------|-------|---------------|
| **p** | AR order | 0, 1, 2, ... | PACF cutoff on differenced series |
| **d** | Differencing order | 0, 1, 2 | ADF test (number of differencings for stationarity) |
| **q** | MA order | 0, 1, 2, ... | ACF cutoff on differenced series |

### 23.3 Special Cases of ARIMA

| Model | ARIMA Notation | Description |
|-------|---------------|-------------|
| White Noise | ARIMA(0,0,0) | No structure |
| Random Walk | ARIMA(0,1,0) | $Y_t = Y_{t-1} + \epsilon_t$ |
| Random Walk with Drift | ARIMA(0,1,0) + c | $Y_t = c + Y_{t-1} + \epsilon_t$ |
| AR(p) | ARIMA(p,0,0) | Pure autoregressive |
| MA(q) | ARIMA(0,0,q) | Pure moving average |
| ARMA(p,q) | ARIMA(p,0,q) | No differencing needed |
| IMA(1,1) | ARIMA(0,1,1) | Simple Exponential Smoothing equivalent |

### 23.4 Parameter Estimation

Parameters are estimated using **Maximum Likelihood Estimation (MLE)**:

$$\hat{\theta} = \arg\max_\theta \prod_{t=1}^{T} f(y_t \mid y_{t-1}, \ldots; \theta)$$

Typically solved using iterative optimization (conditional or exact MLE).

**Python**: `from statsmodels.tsa.arima.model import ARIMA`

---

<a id='24'></a>
## 24. Seasonal ARIMA — SARIMA

### 24.1 Definition

SARIMA extends ARIMA by adding **seasonal AR, differencing, and MA terms**:

$$\text{SARIMA}(p, d, q)(P, D, Q)_m$$

$$\boxed{\Phi_P(B^m) \cdot \Phi_p(B) \cdot (1 - B)^d \cdot (1 - B^m)^D \cdot Y_t = c + \Theta_Q(B^m) \cdot \Theta_q(B) \cdot \epsilon_t}$$

### 24.2 Parameters

**Non-Seasonal Part** (same as ARIMA):

| Parameter | Description |
|-----------|------------|
| $p$ | Non-seasonal AR order |
| $d$ | Non-seasonal differencing order |
| $q$ | Non-seasonal MA order |

**Seasonal Part** (operates at seasonal lag $m$):

| Parameter | Description |
|-----------|------------|
| $P$ | Seasonal AR order (lags $m, 2m, 3m, \ldots$) |
| $D$ | Seasonal differencing order ($\Delta_m$) |
| $Q$ | Seasonal MA order |
| $m$ | Seasonal period (12=monthly, 4=quarterly, 7=daily/weekly, 24=hourly/daily) |

### 24.3 Seasonal Polynomials

$$\Phi_P(B^m) = 1 - \Phi_1 B^m - \Phi_2 B^{2m} - \ldots - \Phi_P B^{Pm}$$
$$\Theta_Q(B^m) = 1 + \Theta_1 B^m + \Theta_2 B^{2m} + \ldots + \Theta_Q B^{Qm}$$

### 24.4 Example: SARIMA(1,1,1)(1,1,1)₁₂

For monthly data ($m = 12$):

$$(1 - \Phi_1 B^{12})(1 - \phi_1 B)(1 - B)(1 - B^{12}) Y_t = (1 + \Theta_1 B^{12})(1 + \theta_1 B) \epsilon_t$$

This means:
- AR(1) on recent lags + SAR(1) on seasonal lag
- First differencing + Seasonal differencing
- MA(1) on recent errors + SMA(1) on seasonal errors

### 24.5 Identifying Seasonal Orders

1. Apply non-seasonal AND seasonal differencing
2. Plot ACF/PACF of the **doubly-differenced** series
3. Look at lags $m, 2m, 3m, \ldots$:
   - ACF cuts off at lag $Qm$ → $Q$
   - PACF cuts off at lag $Pm$ → $P$

**Python**: `from statsmodels.tsa.statespace.sarimax import SARIMAX`

---

<a id='25'></a>
## 25. Box-Jenkins Methodology

The **systematic 4-step approach** to building ARIMA/SARIMA models:

### Step 1: Identification
1. **Plot** the series → Check for trend, seasonality, changing variance
2. **Transform** if needed (log, Box-Cox for variance stabilization)
3. **Difference** to achieve stationarity → Determine $d$ and $D$
4. **Plot ACF/PACF** of the differenced series → Determine $p, q, P, Q$

### Step 2: Estimation
1. Fit the identified model using **Maximum Likelihood Estimation (MLE)**
2. Estimate parameters $\phi_i, \theta_j, \Phi_P, \Theta_Q$

### Step 3: Diagnostic Checking
1. **Residual analysis**:
   - Residuals should be **white noise** (no autocorrelation)
   - Plot ACF of residuals → All lags within confidence bands
   - **Ljung-Box test**: $H_0$: Residuals are white noise (want p > 0.05)
   
   $$Q(H) = n(n+2) \sum_{h=1}^{H} \frac{\hat{\rho}^2(h)}{n-h} \sim \chi^2_{H-p-q}$$
   
2. **Normality check**: Q-Q plot, Shapiro-Wilk test
3. **Homoscedasticity**: Residuals should have constant variance

### Step 4: Forecasting
1. Generate **point forecasts** and **prediction intervals**
2. Prediction intervals widen with forecast horizon (uncertainty grows)

```
Plot → Transform → Difference → ACF/PACF → Fit → Diagnose → Forecast
         ↑                                          |
         └──── If diagnostics fail, revise model ───┘
```

---

<a id='26'></a>
## 26. Auto-ARIMA (pmdarima)

### 26.1 Concept

Automates the Box-Jenkins methodology by **searching over (p, d, q) space** and selecting the model with the best information criterion.

### 26.2 How It Works

1. Uses **KPSS and ADF tests** to determine $d$ (and $D$ for seasonal)
2. Fits models for different $(p, q)$ combinations
3. Selects the model with **lowest AIC** (or BIC or HQIC)
4. Uses **stepwise algorithm** for efficiency (doesn't try all combinations)

### 26.3 Key Parameters

| Parameter | Description | Default |
|-----------|-------------|--------|
| `start_p`, `max_p` | AR order search range | 0, 5 |
| `start_q`, `max_q` | MA order search range | 0, 5 |
| `d` | Differencing order (None = auto) | None |
| `seasonal` | Include seasonal component | True |
| `m` | Seasonal period | 1 (no season) |
| `stepwise` | Use stepwise search (faster) | True |
| `information_criterion` | 'aic', 'bic', 'hqic' | 'aic' |
| `trace` | Print search progress | False |

**Python**: `from pmdarima import auto_arima`

### 26.4 Limitations

- Stepwise search may miss the global optimum
- Relies on information criteria which can disagree
- Doesn't check residual diagnostics automatically
- Always validate with manual ACF/PACF analysis

---

<a id='27'></a>
## 27. Forecasting Evaluation Metrics

### 27.1 Scale-Dependent Metrics

**Mean Absolute Error (MAE)**:

$$\text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|$$

- Easy to interpret: average error in original units
- Less sensitive to outliers than MSE

**Root Mean Squared Error (RMSE)**:

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{t=1}^{n} (y_t - \hat{y}_t)^2}$$

- Penalizes **large errors** more heavily (due to squaring)
- Same units as the data
- More sensitive to outliers than MAE

### 27.2 Percentage-Based Metrics

**Mean Absolute Percentage Error (MAPE)**:

$$\text{MAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \left|\frac{y_t - \hat{y}_t}{y_t}\right|$$

- Scale-independent → can compare across different series
- ⚠️ **Undefined** when $y_t = 0$ (division by zero)
- ⚠️ **Asymmetric**: Penalizes over-forecasting less than under-forecasting

**Symmetric MAPE (sMAPE)**:

$$\text{sMAPE} = \frac{100\%}{n} \sum_{t=1}^{n} \frac{|y_t - \hat{y}_t|}{(|y_t| + |\hat{y}_t|) / 2}$$

- Fixes MAPE's asymmetry problem
- Bounded between 0% and 200%

### 27.3 Scale-Free Metrics

**Mean Absolute Scaled Error (MASE)**:

$$\boxed{\text{MASE} = \frac{\frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|}{\frac{1}{T-m} \sum_{t=m+1}^{T} |y_t - y_{t-m}|}}$$

where denominator = MAE of the **naive seasonal forecast** on the training set.

- **MASE < 1**: Model is better than naive forecast ✅
- **MASE = 1**: Model is as good as naive forecast
- **MASE > 1**: Model is worse than naive forecast ❌
- Works with zeros, no asymmetry, scale-independent
- **Recommended metric** for general time series evaluation

### 27.4 Metric Selection Guide

| Metric | Scale-Free? | Handles Zeros? | Symmetric? | Best For |
|--------|------------|----------------|------------|----------|
| **MAE** | ❌ | ✅ | ✅ | Single series, interpretability |
| **RMSE** | ❌ | ✅ | ✅ | Penalizing large errors |
| **MAPE** | ✅ | ❌ | ❌ | Cross-series comparison (no zeros) |
| **sMAPE** | ✅ | ❌ | ✅ | Better alternative to MAPE |
| **MASE** | ✅ | ✅ | ✅ | **Overall best** — recommended default |

---

<a id='28'></a>
## 28. ML-Based Time Series Forecasting

### 28.1 When ML > Classical Methods

| Scenario | Classical (ARIMA) | ML (XGBoost, etc.) |
|----------|------------------|---------------------|
| **Few features** | ✅ Better | Overkill |
| **Many exogenous features** | Limited | ✅ Better |
| **Non-linear relationships** | ❌ Limited | ✅ Better |
| **Small data** | ✅ Better (fewer params) | ❌ Overfits |
| **Large data** | Slow / limited | ✅ Scales well |
| **Interpretability** | ✅ Clear math | Black-box |
| **Multiple series** | Fit per series | ✅ Can learn across series |

### 28.2 ML Feature Engineering for Time Series

Transform time series into a **tabular supervised learning problem**:

```
Original Series: [y1, y2, y3, y4, y5, y6, y7]

Tabular Form (with lag=3):
| lag_3 | lag_2 | lag_1 | target |
|-------|-------|-------|--------|
|  y1   |  y2   |  y3   |   y4   |
|  y2   |  y3   |  y4   |   y5   |
|  y3   |  y4   |  y5   |   y6   |
|  y4   |  y5   |  y6   |   y7   |
```

### 28.3 Feature Categories

1. **Lag features**: $y_{t-1}, y_{t-2}, \ldots, y_{t-k}$
2. **Rolling statistics**: rolling mean, std, min, max over windows
3. **Calendar features**: hour, day, month, is_weekend, is_holiday
4. **Cyclical encodings**: sin/cos of calendar features
5. **Exogenous variables**: weather, promotions, economic indicators
6. **Difference features**: $y_t - y_{t-1}$, percentage changes

### 28.4 XGBoost for Time Series

XGBoost is popular for time series because:
- Handles non-linear relationships
- Robust to outliers
- Can incorporate many exogenous features
- Feature importance helps identify key drivers
- Fast training and inference

### 28.5 Critical: Validation Strategy

**NEVER use random train-test split for time series!**

- Use **Walk-Forward Validation** (Section 15)
- Ensure **no future leakage** in features
- Lag features must only use **past** data at prediction time

### 28.6 Multi-Step Forecasting Strategies

| Strategy | Description | Pros/Cons |
|----------|-------------|----------|
| **Recursive** | Predict $t+1$, use it to predict $t+2$, etc. | Simple but error accumulates |
| **Direct** | Train separate model for each horizon | No error propagation but expensive |
| **Multi-output** | Single model predicts all horizons simultaneously | Balanced approach |

---

<a id='29'></a>
## 29. Facebook Prophet

### 29.1 Overview

Prophet is an **additive regression model** developed by Meta (Facebook) for business time series forecasting:

$$\boxed{y(t) = g(t) + s(t) + h(t) + \epsilon_t}$$

| Component | Symbol | Description |
|-----------|--------|------------|
| **Trend** | $g(t)$ | Piecewise linear or logistic growth |
| **Seasonality** | $s(t)$ | Fourier series for periodic patterns |
| **Holidays** | $h(t)$ | User-specified holiday effects |
| **Error** | $\epsilon_t$ | Normally distributed noise |

### 29.2 Trend Models

**Piecewise Linear** (default):
$$g(t) = (k + \mathbf{a}(t)^T \boldsymbol{\delta}) \cdot t + (m + \mathbf{a}(t)^T \boldsymbol{\gamma})$$

where $k$ = base growth rate, $\boldsymbol{\delta}$ = rate adjustments at **changepoints**, $m$ = offset.

**Logistic Growth** (for saturating trends):
$$g(t) = \frac{C}{1 + \exp(-(k + \mathbf{a}(t)^T \boldsymbol{\delta})(t - (m + \mathbf{a}(t)^T \boldsymbol{\gamma})))}$$

where $C$ = carrying capacity (saturation point).

### 29.3 Seasonality via Fourier Series

$$s(t) = \sum_{n=1}^{N} \left(a_n \cos\left(\frac{2\pi n t}{P}\right) + b_n \sin\left(\frac{2\pi n t}{P}\right)\right)$$

- $P$ = period (365.25 for yearly, 7 for weekly)
- $N$ = number of Fourier terms (higher = more flexibility, risk of overfitting)
- Default: $N = 10$ for yearly, $N = 3$ for weekly

### 29.4 Key Parameters

| Parameter | Description | Effect |
|-----------|-------------|--------|
| `changepoint_prior_scale` | Flexibility of trend changepoints | Higher = more flexible trend |
| `seasonality_prior_scale` | Flexibility of seasonal component | Higher = stronger seasonal swings |
| `holidays_prior_scale` | Flexibility of holiday effects | Higher = stronger holiday impact |
| `seasonality_mode` | 'additive' or 'multiplicative' | Match data pattern |
| `changepoint_range` | Proportion of history for changepoints | Default 0.8 (first 80%) |

### 29.5 Strengths & Limitations

| Strengths | Limitations |
|-----------|------------|
| ✅ Easy to use (minimal config) | ❌ No native support for exogenous features |
| ✅ Handles missing data & outliers | ❌ Univariate only (but add regressors manually) |
| ✅ Automatic changepoint detection | ❌ Can overfit with short series |
| ✅ Built-in holiday handling | ❌ Not designed for high-frequency data |
| ✅ Uncertainty intervals out-of-box | ❌ Bayesian approach can be slow |

**Python**: `from prophet import Prophet`

---

<a id='30'></a>
## 30. ARIMA vs ML — Comparison Matrix

| Criterion | ARIMA/SARIMA | ML (XGBoost, etc.) | Prophet |
|-----------|-------------|---------------------|--------|
| **Data requirement** | Can work with small data | Needs more data | Moderate |
| **Exogenous features** | SARIMAX supports some | Handles many naturally | Add as regressors |
| **Non-linearity** | ❌ Linear only | ✅ Handles well | Limited |
| **Seasonality** | Single seasonal period | Multiple via features | Multiple Fourier |
| **Stationarity needed** | ✅ Required | ❌ Not required | ❌ Not required |
| **Feature engineering** | Minimal (auto lags) | Extensive (manual) | Automatic |
| **Interpretability** | ✅ High (math-based) | ⚠️ Medium (SHAP) | ✅ High (decomposition) |
| **Multi-step forecast** | Native | Recursive/Direct | Native |
| **Uncertainty estimates** | ✅ Prediction intervals | ❌ Needs extra work | ✅ Built-in |
| **Speed** | Fast for single series | Fast with tabular | Slower (Bayesian) |
| **Multiple series** | Fit per series | ✅ Global model | Fit per series |
| **Best for** | Univariate, linear, short | Feature-rich, non-linear | Business forecasting |

### Rule of Thumb

1. **Start simple**: Try naive forecast → ARIMA → ML → Prophet
2. **Always compare** against a naive baseline ($\hat{y}_t = y_{t-m}$)
3. **Ensemble** different approaches for best results

---

<a id='31'></a>
## 31. Key Takeaways & Interview Questions

### 📝 Key Takeaways

1. **Stationarity** requires constant mean, variance, and autocovariance — test with ADF + KPSS together.
2. **Differencing** removes trend ($d$) and seasonality ($D$); log/Box-Cox stabilizes variance.
3. **ACF** identifies MA order (cutoff at $q$); **PACF** identifies AR order (cutoff at $p$).
4. **ARIMA(p,d,q)** is the workhorse for univariate forecasting; **SARIMA** adds seasonal structure.
5. **Box-Jenkins** methodology: Identify → Estimate → Diagnose → Forecast.
6. **MASE** is the recommended evaluation metric — scale-free, handles zeros, symmetric.
7. **ML forecasting** converts time series to tabular via lag features and rolling statistics.
8. **Walk-Forward CV** is mandatory — never use random splits for time series.
9. **Prophet** is great for quick business forecasting with holidays and changepoints.
10. **Ensemble** classical + ML methods for robust forecasting.

### 🎯 Interview Questions

1. **What is stationarity and why does ARIMA need it?**
2. **Explain the difference between ADF and KPSS tests.**
3. **How do you read ACF and PACF plots to choose p, d, q?**
4. **What's the difference between AR and MA models?**
5. **Explain SARIMA parameters $(p,d,q)(P,D,Q)_m$.**
6. **Why can't you use K-Fold CV for time series?**
7. **What is MASE and why is it preferred over MAPE?**
8. **How do you convert a time series problem to a tabular ML problem?**
9. **When would you use XGBoost vs ARIMA for forecasting?**
10. **What is the Box-Jenkins methodology?**
11. **Explain the Ljung-Box test. What does it check?**
12. **What is a unit root and what does it mean for forecasting?**
13. **How does Prophet handle trend changepoints?**
14. **What happens if you over-difference a series?**
15. **Explain cyclical encoding (sin/cos) for time features.**

---

🎉 **End of Master Time Series Theory** — Now practice by writing code for each section!